In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
storage_account_name = "silveradlsstorage"
silver_base = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
gold_base = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"
silver_path = f"{silver_base}/retail_delta_clean"
gold_path = f"{gold_base}/retail_dw"
batch_id = "manual-20260602-scd-001"

# Configure ADLS Gen2 OAuth service principal authentication
sp_client_id     = dbutils.secrets.get(scope="retail-adls-kv-scope", key="adls-sp-client-id")
sp_client_secret = dbutils.secrets.get(scope="retail-adls-kv-scope", key="adls-sp-client-secret")
sp_tenant_id     = dbutils.secrets.get(scope="retail-adls-kv-scope", key="adls-sp-tenant-id")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", sp_client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", sp_client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{sp_tenant_id}/oauth2/token")



In [0]:
products_source_df = (
    spark.read.format("delta")
    .load(f"{silver_path}/products")
    .select(
        F.col("productid").alias("ProductId"),
        F.col("productname").alias("ProductName"),
        F.col("category").alias("Category"),
        F.col("subcategory").alias("Subcategory"),
        F.col("brand").alias("Brand"),
        F.col("unitprice").alias("UnitPrice"),
        F.col("productid").cast("long").alias("ProductKey")
    )
    .dropDuplicates(["ProductId"])
    .withColumn("GoldProcessedAtUtc", F.current_timestamp())
)
dim_product_path = f"{gold_path}/dim_product"
dim_product_delta = DeltaTable.forPath(spark, dim_product_path)
(
    dim_product_delta.alias("target")
    .merge(
        products_source_df.alias("source"),
        "target.ProductId = source.ProductId"
    )
    .whenMatchedUpdate(set={
        "ProductName": "source.ProductName",
        "Category": "source.Category",
        "Subcategory": "source.Subcategory",
        "Brand": "source.Brand",
        "UnitPrice": "source.UnitPrice",
        "GoldProcessedAtUtc": "source.GoldProcessedAtUtc"
    })
    .whenNotMatchedInsert(values={
        "ProductKey": "source.ProductKey",
        "ProductId": "source.ProductId",
        "ProductName": "source.ProductName",
        "Category": "source.Category",
        "Subcategory": "source.Subcategory",
        "Brand": "source.Brand",
        "UnitPrice": "source.UnitPrice",
        "EffectiveStartDate": "current_date()",
        "EffectiveEndDate": "cast('9999-12-31' as date)",
        "IsCurrent": "true",
        "GoldProcessedAtUtc": "source.GoldProcessedAtUtc"
    })
    .execute()
)



In [0]:
display(spark.read.format("delta").load(dim_product_path))


In [0]:
customers_source_df = (
    spark.read.format("delta")
    .load(f"{silver_path}/customers")
    .select(
        F.col("customerid").alias("CustomerId"),
        F.col("firstname").alias("FirstName"),
        F.col("lastname").alias("LastName"),
        F.col("email").alias("Email"),
        F.col("phone").alias("Phone"),
        F.col("city").alias("City"),
        F.col("stateprovince").alias("State"),
        F.col("country").alias("Country")
    )
    .dropDuplicates(["CustomerId"])
    .withColumn(
        "HashDiff",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("FirstName"), F.lit("")),
                F.coalesce(F.col("LastName"), F.lit("")),
                F.coalesce(F.col("Email"), F.lit("")),
                F.coalesce(F.col("Phone"), F.lit("")),
                F.coalesce(F.col("City"), F.lit("")),
                F.coalesce(F.col("State"), F.lit("")),
                F.coalesce(F.col("Country"), F.lit(""))
            ),
            256
        )
    )
)


In [0]:
dim_customer_path = f"{gold_path}/dim_customer"
dim_customer_df = spark.read.format("delta").load(dim_customer_path)
if "HashDiff" not in dim_customer_df.columns:
    dim_customer_df = dim_customer_df.withColumn(
        "HashDiff",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("FirstName"), F.lit("")),
                F.coalesce(F.col("LastName"), F.lit("")),
                F.coalesce(F.col("Email"), F.lit("")),
                F.coalesce(F.col("Phone"), F.lit("")),
                F.coalesce(F.col("City"), F.lit("")),
                F.coalesce(F.col("State"), F.lit("")),
                F.coalesce(F.col("Country"), F.lit(""))
            ),
            256
        )
    )
    (
        dim_customer_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(dim_customer_path)
    )


In [0]:
dim_customer_delta = DeltaTable.forPath(spark, dim_customer_path)
(
    dim_customer_delta.alias("target")
    .merge(
        customers_source_df.alias("source"),
        """
        target.CustomerId = source.CustomerId
        AND target.IsCurrent = true
        AND target.HashDiff <> source.HashDiff
        """
    )
    .whenMatchedUpdate(set={
        "IsCurrent": "false",
        "EffectiveEndDate": "current_date()",
        "GoldProcessedAtUtc": "current_timestamp()"
    })
    .execute()
)


In [0]:
current_dim_customer_df = spark.read.format("delta").load(dim_customer_path)
records_to_insert_df = (
    customers_source_df.alias("source")
    .join(
        current_dim_customer_df.filter(F.col("IsCurrent") == True).alias("target"),
        F.col("source.CustomerId") == F.col("target.CustomerId"),
        "left"
    )
    .filter(F.col("target.CustomerId").isNull())
    .select("source.*")
    .withColumn("CustomerKey", F.monotonically_increasing_id() + 100000)
    .withColumn("EffectiveStartDate", F.current_date())
    .withColumn("EffectiveEndDate", F.lit("9999-12-31").cast("date"))
    .withColumn("IsCurrent", F.lit(True))
    .withColumn("GoldProcessedAtUtc", F.current_timestamp())
)
(
    records_to_insert_df
    .select(
        "CustomerKey",
        "CustomerId",
        "FirstName",
        "LastName",
        "Email",
        "Phone",
        "City",
        "State",
        "Country",
        "EffectiveStartDate",
        "EffectiveEndDate",
        "IsCurrent",
        "GoldProcessedAtUtc",
        "HashDiff"
    )
    .write
    .format("delta")
    .mode("append")
    .save(dim_customer_path)
)


In [0]:
display(
    spark.read.format("delta")
    .load(dim_customer_path)
    .orderBy("CustomerId", "EffectiveStartDate")
)
